# Vietnamese Voice Cloning Studio — Fine-tune trên giọng của bạn (Google Colab)

Notebook này chạy trên **GPU T4 miễn phí của Colab** — máy Mac cá nhân không train được
vì `scripts/train.py` cần GPU CUDA hỗ trợ FP16. Đây là con đường thực tế duy nhất để
fine-tune miễn phí mà không cần mua phần cứng.

## Quy trình tổng quan (quan trọng — đọc trước khi chạy)
**Bước xử lý dữ liệu (cắt đoạn, chuẩn hoá) chạy TRÊN MÁY MAC, không chạy trong Colab.**
Lý do: bản `scripts/data_prep.py` mới nhất (đã sửa lỗi cắt cứng giữa từ) hiện chỉ có trên
máy bạn, chưa được đẩy lên GitHub — nếu để Colab tự `git clone`, nó sẽ tải bản CŨ còn lỗi.
Vì `data_prep.py` không cần GPU, chạy trên Mac vừa nhanh vừa dùng đúng bản đã sửa. Colab
chỉ đảm nhận phần THỰC SỰ cần GPU: `train.py`.

```
[Mac — không cần GPU]                    [Colab — cần GPU T4]
data/raw/ (audio thật)                    
   → scripts/data_prep.py                 
   → data/processed/ + metadata.csv        
   → (bạn điền transcript thủ công)        
   → nén & tải lên Google Drive    ────►   mount Drive → scripts/train.py
                                            → checkpoint lưu vào Drive
   (tải checkpoint về máy)         ◄────   
   → dùng trong app.py local
```

## Trước khi mở notebook này, hãy làm trên máy Mac
1. Đặt file audio giọng thật (30–60 phút, 1 người nói, sạch) vào `data/raw/` —
   xem `data/raw/README.md` trong repo.
2. Chạy: `python scripts/data_prep.py --input_dir data/raw --output_dir data/processed`
3. Mở `data/metadata/metadata.csv`, nghe từng đoạn, điền transcript chính xác vào cột `text`.
4. Nén hai thứ này thành 1 file zip: `data/processed/` và `data/metadata/metadata.csv`
   (ví dụ: `zip -r vvcs_data.zip data/processed data/metadata`).
5. Tải `vvcs_data.zip` lên Google Drive, thư mục gốc (`My Drive`) hoặc bất kỳ đâu bạn nhớ.

Xong các bước trên thì mới chạy các cell bên dưới.

## Bước 1 — Kiểm tra GPU
Nếu chưa bật: menu `Runtime` → `Change runtime type` → chọn **T4 GPU** → Save → chạy lại cell này.

In [ ]:
import torch
has_gpu = torch.cuda.is_available()
print("CUDA available:", has_gpu)
if has_gpu:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("\n⚠️  CHƯA CÓ GPU. Runtime > Change runtime type > T4 GPU > Save, ")
    print("    rồi Runtime > Restart session, sau đó chạy lại cell này.")
assert has_gpu, "Cần bật GPU trước khi tiếp tục (xem hướng dẫn ở trên)."

## Bước 2 — Mount Google Drive + giải nén dữ liệu đã chuẩn bị trên Mac

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/vietnamese-voice-cloning-studio'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Thư mục dự án trên Drive: {PROJECT_DIR}")

In [ ]:
# Đổi đường dẫn này đúng vị trí bạn đã tải vvcs_data.zip lên Drive ở bước chuẩn bị.
ZIP_PATH = '/content/drive/MyDrive/vvcs_data.zip'

import pathlib
assert pathlib.Path(ZIP_PATH).exists(), (
    f"Không tìm thấy {ZIP_PATH}. Hãy tải vvcs_data.zip (nén từ data/processed/ + "
    f"data/metadata/metadata.csv trên Mac) lên Drive rồi sửa lại ZIP_PATH ở trên."
)
!mkdir -p "{PROJECT_DIR}"
!unzip -o -q "{ZIP_PATH}" -d "{PROJECT_DIR}"
print("Đã giải nén vào:", PROJECT_DIR)
!find "{PROJECT_DIR}/data" -maxdepth 2

## Bước 3 — Tải mã nguồn dự án (chỉ dùng phần train, KHÔNG dùng data_prep.py ở đây) + cài thư viện

In [ ]:
!git clone --depth 1 https://github.com/Thanh-Nguyen2206/vietnamese-voice-cloning-studio.git /content/vvcs
%cd /content/vvcs
!pip install -q -r requirements.txt

## Bước 4 — Kiểm tra dữ liệu & transcript trước khi train
Xác nhận số đoạn tìm được và không còn transcript trống — tránh chạy nhầm dữ liệu thiếu.

In [ ]:
import csv
meta_path = f'{PROJECT_DIR}/data/metadata/metadata.csv'
with open(meta_path, encoding='utf-8') as f:
    rows = list(csv.DictReader(f, delimiter='|'))
missing = [r['audio_file'] for r in rows if not r.get('text', '').strip()]
total_min = sum(float(r.get('duration_sec', 0)) for r in rows) / 60
print(f"Tổng {len(rows)} đoạn ({total_min:.1f} phút). Còn thiếu transcript: {len(missing)}")
if missing:
    print("⚠️  Ví dụ còn thiếu (tối đa 10):", missing[:10])
    print("    Quay lại Mac, điền nốt metadata.csv, nén lại, tải lên Drive, chạy lại Bước 2.")
else:
    print("✅ Đã điền đủ transcript cho mọi đoạn — sẵn sàng train!")
assert not missing, "Còn transcript trống — điền xong rồi mới train (xem trên)."

## Bước 5 — Tạo config trỏ tới Google Drive (để checkpoint không mất khi ngắt phiên)

In [ ]:
import yaml
with open('configs/train_config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['data']['processed_dir'] = f'{PROJECT_DIR}/data/processed'
cfg['data']['metadata_path'] = f'{PROJECT_DIR}/data/metadata/metadata.csv'
cfg['checkpoint']['save_dir'] = f'{PROJECT_DIR}/checkpoints'
cfg['logging']['log_dir'] = f'{PROJECT_DIR}/logs'

with open('configs/train_config_colab.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)

print("Đã tạo configs/train_config_colab.yaml — dữ liệu/checkpoint/log đều trỏ vào Drive.")
print(f"  epochs={cfg['training']['epochs']}  batch={cfg['training']['batch_size']}"
      f"  grad_accum={cfg['training']['gradient_accumulation_steps']}"
      f"  lr={cfg['training']['learning_rate']}")

## Bước 6 — Train

**Về thời gian & Colab miễn phí:** với 30–60 phút dữ liệu, cấu hình mặc định
(`epochs: 100`) có thể mất NHIỀU giờ trên T4 — nhiều hơn một phiên Colab miễn phí
thường cho phép (~90 phút không thao tác là bị ngắt, tối đa ~12 giờ liên tục). Điều
này ổn: checkpoint tự lưu vào Drive mỗi 500 bước (`save_every_n_steps`). `train.py`
không tự nạp checkpoint để tránh dùng nhầm dữ liệu/model cũ. Nếu bị ngắt phiên, chạy lại
từ Bước 3 rồi thêm `--resume latest` vào lệnh dưới để tiếp tục checkpoint đã xác minh
đúng chỗ đã dừng, không train lại từ đầu.

Bạn cũng có thể dừng sớm (Runtime > Interrupt execution) và **thử ngay checkpoint hiện
có** trong app web local (xem Bước 7) — không bắt buộc phải chạy đủ 100 epoch mới nghe
được kết quả. Nhiều recipe fine-tune giọng cá nhân cho F5-TTS đã nghe được rõ giọng chỉ
sau vài nghìn bước, không cần train hết cấu hình mặc định.

In [ ]:
# Lần đầu: bỏ --resume latest. Chỉ thêm cờ này sau khi phiên trước đã tạo checkpoint.
!python scripts/train.py --config configs/train_config_colab.yaml # --resume latest

## Bước 7 — Mang checkpoint về máy để dùng trong web app local

1. Trên Google Drive, vào `vietnamese-voice-cloning-studio/checkpoints/`, tìm thư mục
   `step_XXXXXXX` mới nhất (số càng lớn càng train nhiều).
2. Tải NGUYÊN thư mục đó về máy Mac, đặt vào đúng vị trí:
   `vietnamese-voice-cloning-studio/checkpoints/step_XXXXXXX/` (giữ nguyên tên file
   `model.pt` bên trong).
3. Chạy lại `python app.py` trên máy — checkpoint sẽ **tự động xuất hiện** trong danh
   sách chọn mô hình với tên **"Giọng cá nhân (tinh chỉnh, step ...)"**, sẵn sàng so
   sánh với F5-TTS gốc và các engine khác ngay trên giao diện so sánh 6 mô hình.

Không cần tải `optimizer.pt` hay `training_state.pt` về máy — hai file đó chỉ cần nếu
muốn TIẾP TỤC train (giữ lại trên Drive), không cần cho việc suy luận/nghe thử.